## Predict Hotel Cancellations

In [2]:
import pandas as pd
import numpy as np

### Import and Inspect

The file contains a subset of a [real-world dataset](https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand) containing reservation and cancellation data for a resort hotel. 

The goal is to build and train a neural network to predict if a customer will cancel their hotel booking reservation based on data including the booking dates, average daily cost, number of adults/children/babies, duration of stay etc.

In [3]:
hotels = pd.read_csv('resort_hotel_bookings.csv')

In [17]:
hotels.head()

,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,0,342,2015,July,27,1,0,0,2,0.0,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,0,737,2015,July,27,1,0,0,2,0.0,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,0,7,2015,July,27,1,0,1,1,0.0,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,0,13,2015,July,27,1,0,1,1,0.0,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,0,14,2015,July,27,1,0,2,2,0.0,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


<details><summary style="display:list-item; font-size:16px; color:blue;">Here's a quick summary of the columns</summary>

- **is_canceled**: Whether the booking was canceled (1) or kept (0)
- **lead_time**: Number of days between booking date and arrival date
- **arrival_date_year**: Year of arrival date
- **arrival_date_month**: Month of arrival date
- **arrival_date_week_number**: Week number of arrival date
- **arrival_date_day_of_month**: Day of the month of arrival date
- **stays_in_weekend_nights**: Number of weekend nights booked (Sat-Sun)
- **stay_in_week_nights**: Number of weekday nights booked (Mon-Fri)
- **adults**: Number of adults
- **children**: Number of children
- **babies**: Number of babies
- **meal**: Type of meal booked (Undefined/SC, BB, HB, or FB)
- **country**: Country of origin of the booker
- **market_segment**: Market segment (TA - travel agent, TO - tour operators)
- **distribution_channel**: Booking distribution channel (TA - travel agent, TO - tour operators)
- **is_repeated_guest**: Is this a repeated guest (1) or not (0)
- **previous_cancellations**: The number of previous bookings canceled by the customer
- **previous_bookings_not_canceled**: The number of previous bookings not canceled by the customer
- **reserved_room_type**: Room type reserved
- **assigned_room_type**: Type of assigned room booked
- **booking_changes**: Number of booking changes or modifications
- **deposit_type**: Type of deposit to guarantee booking (No Deposit, Non Refund, or Refundable)
- **agent**: ID of the travel agency that made the booking
- **company**: ID of the company that made the booking
- **days_in_waiting_list**: Number of days booking was waitlisted before confirmation
- **customer_type**: The customer type of booking (Contract, Group, Transient, or Transient-party)
- **adr**: The average daily rate (cost) of the booking
- **required_car_parking_spaces**: Number of parking spaces requested by the customer
- **total_of_special_requests**: Number of special requests by the customer
- **reservation_status**: The last reservation status (Canceled, Check-Out, No-Show)
- **reservation_status_date**: The date of the last reservation status

In [3]:
hotels.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40060 entries, 0 to 40059
Data columns (total 31 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   is_canceled                     40060 non-null  int64  
 1   lead_time                       40060 non-null  int64  
 2   arrival_date_year               40060 non-null  int64  
 3   arrival_date_month              40060 non-null  object 
 4   arrival_date_week_number        40060 non-null  int64  
 5   arrival_date_day_of_month       40060 non-null  int64  
 6   stays_in_weekend_nights         40060 non-null  int64  
 7   stays_in_week_nights            40060 non-null  int64  
 8   adults                          40060 non-null  int64  
 9   children                        40060 non-null  float64
 10  babies                          40060 non-null  int64  
 11  meal                            40060 non-null  object 
 12  country                         

<details><summary style="display:list-item; font-size:16px; color:blue;">What do we notice about the dataset under inspection?</summary>

There are 31 columns and 40,060 total observations in our dataset. The majority of columns do not have missing values.

However, we do notice that: 
- the `agent` and `company` columns seem to have missing values that need to be addressed
- the `country` column has a couple of missing values as well

There are a variety of data types represented. To work with a neural network, we need to address any non-numeric columns in our data preparation.

In [4]:
hotels.is_canceled.value_counts()

is_canceled
0    28938
1    11122
Name: count, dtype: int64

In [5]:
hotels.is_canceled.value_counts(1)

is_canceled
0    0.722366
1    0.277634
Name: proportion, dtype: float64

<details><summary style="display:list-item; font-size:16px; color:blue;">What do we notice about the number of cancellations?</summary>

The number of cancellations is much lower than the number of non-cancellations (27.8% canceled vs 72.2% did not cancel). 

We'll need to take this imbalance into account when we evaluate our model. For example, a naive model could simply predict every booking will **not be canceled** and achieve a decent accuracy of 72.2%.

The `reservation_status` column tells us if the booking was canceled while also telling us if the customer was a no-show.

We need to be sure to exclude this column from the training set, otherwise this information will be _leaked_ to our model resulting in inaccurate performance.

In [6]:
hotels.reservation_status.value_counts()

reservation_status
Check-Out    28938
Canceled     10831
No-Show        291
Name: count, dtype: int64

In [7]:
hotels.reservation_status.value_counts(1)

reservation_status
Check-Out    0.722366
Canceled     0.270369
No-Show      0.007264
Name: proportion, dtype: float64

<details><summary style="display:list-item; font-size:16px; color:blue;">What do we notice about the reservation_status column?</summary>

The number of no-shows is extremely small and consists of only 291 (or 0.7%) of observations in the dataset.

It's important to understand how different columns interact with cancellations to guide our model structure. For example, cancellations might be higher in the summer months (June - September) and lower in the winter months (November - January).

In [8]:
hotels.groupby('arrival_date_month')['is_canceled'].mean().sort_values()

arrival_date_month
January      0.148199
November     0.189167
March        0.228717
December     0.238293
February     0.256204
October      0.275105
May          0.287721
April        0.293433
July         0.314017
September    0.323681
June         0.330706
August       0.334491
Name: is_canceled, dtype: float64

<details><summary style="display:list-item; font-size:16px; color:blue;">What do we notice about the percentage of cancellations by month?</summary>

It looks like our intuition was correct! Winter and spring have the lowest cancellation percentages, while summer and fall have the highest. This information can be very useful for our model!

### Data Cleaning and Preparation

encode categorical data for use in our neural networks.

In [4]:
object_columns = ['arrival_date_month', 'meal', 'country', 'market_segment', 
                  'distribution_channel', 'reserved_room_type', 'assigned_room_type',
                  'deposit_type', 'customer_type']
hotels[object_columns].head()

,arrival_date_month,meal,country,market_segment,distribution_channel,reserved_room_type,assigned_room_type,deposit_type,customer_type
0,July,BB,PRT,Direct,Direct,C,C,No Deposit,Transient
1,July,BB,PRT,Direct,Direct,C,C,No Deposit,Transient
2,July,BB,GBR,Direct,Direct,A,C,No Deposit,Transient
3,July,BB,GBR,Corporate,Corporate,A,A,No Deposit,Transient
4,July,BB,GBR,Online TA,TA/TO,A,A,No Deposit,Transient


Typically, we don't want to use every column in training. For example, we may want to drop columns with many missing values or columns that are irrelevant to our prediction task.

In [5]:
drop_columns = ['country', 'agent', 'company', 'reservation_status_date',
                'arrival_date_week_number', 'arrival_date_day_of_month', 'arrival_date_year']

In [6]:
hotels = hotels.drop(labels=drop_columns, axis=1)

<details><summary style="display:list-item; font-size:16px; color:blue;">Hint: Drop columns in the dataset</summary> 
    
Why we chose these columns:

- `country` - there are many countries that only appear a handful of times in the dataset which may make our model less generalizable and even discriminate against customers based on their country
- `agent` - similar to `country`, there are many agents that only appear a handful of times which may make our model less generalizable (and there are many missing values!)
- `company` - similar to `agent`, there are many companies that only appear a handful of times which may make our model less generalizable (and there are many missing values!)
- `reservation_status_date` - tells us the date of the latest status change of the reservation which shouldn't be helpful and if anything may leak data
- `arrival_date_week_number` - tells us the week of the year which may be too specific and prone to overfitting
- `arrival_date_day_of_month` - tells us the day of the month which may be too specific and prone to overfitting
- `arrival_date_year` - tells us the year of the booking which may not be helpful to predict future years

</details>

Next, let's encode the `meal` column which tells us which type of meal(s) the customer booked: 

- `Undefined` and `SC` correspond to no meal packages, encode to 0
- `BB` corresponds to breakfast only, to 1
- `HB` (half board) corresponds to breakfast + lunch or dinner, to 2
- `FB` (full board) corresponds to breakfast, lunch, and dinner, to 3.

In [8]:
hotels['meal'] = hotels['meal'].replace({'Undefined':0, 'SC':0, 'BB':1, 'HB':2, 'FB':3})

prepare the rest of the categorical columns using one-hot encoding. 

In [9]:
one_hot_columns = ['arrival_date_month', 'market_segment', 'distribution_channel', 'reserved_room_type', 
                   'assigned_room_type', 'deposit_type', 'customer_type']
hotels = pd.get_dummies(hotels, columns=one_hot_columns, dtype=int)

The cleaned DataFrame now has 67 columns due to the additional columns created using one-hot encoding.

### Create Training and Testing Sets

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim

In [11]:
# Remove target columns
remove_cols = ['is_canceled', 'reservation_status']

train_features = [x for x in hotels.columns if x not in remove_cols]

In [13]:
X = torch.tensor(hotels[train_features].values, dtype=torch.float)
y = torch.tensor(hotels.is_canceled.values, dtype=torch.float).view(-1, 1)

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=42)
X_train.shape, X_test.shape

(torch.Size([32048, 65]), torch.Size([8012, 65]))

Our training set `X_train` have 65 training features.

### Train a Neural Network for Binary Classification

In [15]:
torch.manual_seed(42)

model = nn.Sequential(nn.Linear(65, 36), # 65 training features
                     nn.ReLU(),
                     nn.Linear(36, 18), # first hidden layer with 36 nodes
                     nn.ReLU(),
                     nn.Linear(18, 1), # second hidden layer with 18 nodes
                     nn.Sigmoid())

In [16]:
loss = nn.BCELoss() # binary cross-entropy
optimizer = optim.Adam(model.parameters(), lr=0.005)

In [24]:
from sklearn.metrics import accuracy_score

num_epochs = 1000
for epoch in range(num_epochs):
    predictions = model(X_train)
    BCELoss = loss(predictions, y_train)
    BCELoss.backward()
    optimizer.step()
    optimizer.zero_grad()
    
    if (epoch + 1) % 100 == 0:
        predicted_labels = (predictions>=0.5).int()
        accuracy = accuracy_score(y_train, predicted_labels)
        print(f'Epoch [{epoch+1}/{num_epochs}], BCELoss: {BCELoss:.4f}, Accuracy: {accuracy:.4f}')

Epoch [100/1000], BCELoss: 0.3502, Accuracy: 0.8370


Epoch [200/1000], BCELoss: 0.3441, Accuracy: 0.8389


Epoch [300/1000], BCELoss: 0.3425, Accuracy: 0.8367


Epoch [400/1000], BCELoss: 0.3359, Accuracy: 0.8416


Epoch [500/1000], BCELoss: 0.3318, Accuracy: 0.8433


Epoch [600/1000], BCELoss: 0.3287, Accuracy: 0.8430


Epoch [700/1000], BCELoss: 0.3288, Accuracy: 0.8445


Epoch [800/1000], BCELoss: 0.3255, Accuracy: 0.8460


Epoch [900/1000], BCELoss: 0.3234, Accuracy: 0.8467


Epoch [1000/1000], BCELoss: 0.3341, Accuracy: 0.8437


In [25]:
model.eval() # evaluation mode
with torch.no_grad(): # Turn off gradient calculations
    test_predictions = model(X_test)
    test_predictions_labels = (test_predictions>=0.5).int() # Convert to binary labels using threshold

Recall that the number of cancellations is much lower than the number of non-cancellations (27.8% canceled vs 72.2% did not cancel). 

In [27]:
from sklearn.metrics import classification_report

accuracy = accuracy_score(y_test, test_predictions_labels)
print(accuracy)
rep = classification_report(y_test, test_predictions_labels)
print(rep)

0.8308786819770344
              precision    recall  f1-score   support

         0.0       0.89      0.87      0.88      5798
         1.0       0.68      0.73      0.71      2214

    accuracy                           0.83      8012
   macro avg       0.79      0.80      0.79      8012
weighted avg       0.84      0.83      0.83      8012



Overall, the model seems to perform reasonably well at predicting hotel cancellations!

The model has an overall accuracy of 83%, indicating that 83% of our model's predictions are correct.
The precision score tells us that when our model predicts a cancellation, it is correct ~72% of the time.
The recall score tells us that our model captures about 68% of the actual cancellations in our data. 

In future research, we could improve the model by performing a more in-depth analysis of the features and doing a more robust feature selection process (like gathering more features or dropping less useful features). 

Furthermore, we could modify the neural network architecture by changing the number of nodes across the hidden layers, trying out different activation functions and optimizers, adding more hidden layers, or training on additional epochs.

### Train a Neural Network for Multiclass Classification

now extend our binary classification task to multiclass by attempting to also predict customers who **no-showed** within the `reservation_status` column.

If a hotel can accurately predict no-shows, they can reach out ahead of time to customers who are at high risk of not-showing to their reservation.

label encode the three categories

In [18]:
hotels['reservation_status'] = hotels['reservation_status'].replace({'Check-Out':2, 'Canceled':1, 'No-Show': 0})

In [36]:
X = torch.tensor(hotels[train_features].values, dtype=torch.float)
y = torch.tensor(hotels.reservation_status.values, dtype=torch.long)

In [37]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=42)

In [31]:
torch.manual_seed(42)

multiclass_model = nn.Sequential(
     nn.Linear(65, 65),
     nn.ReLU(),
     nn.Linear(65, 36),
     nn.ReLU(),
     nn.Linear(36, 3)) # 3 nodes corresponding to each of the categories in reservation_status

In [34]:
loss = nn.CrossEntropyLoss()
optimizer = optim.Adam(multiclass_model.parameters(), lr=0.01)

In [38]:
num_epochs = 500
for epoch in range(num_epochs):
    predictions = multiclass_model(X_train)
    CELoss = loss(predictions, y_train)
    CELoss.backward()
    optimizer.step()
    optimizer.zero_grad()
    
    if (epoch + 1) % 100 == 0:
        predicted_labels = torch.argmax(predictions, dim=1)
        accuracy = accuracy_score(y_train, predicted_labels)
        print(f'Epoch [{epoch+1}/{num_epochs}], CELoss: {CELoss:.4f}, Accuracy: {accuracy:.4f}')

Epoch [100/500], CELoss: 0.4142, Accuracy: 0.8215


Epoch [200/500], CELoss: 0.3927, Accuracy: 0.8257


Epoch [300/500], CELoss: 0.3832, Accuracy: 0.8250


Epoch [400/500], CELoss: 0.3592, Accuracy: 0.8398


Epoch [500/500], CELoss: 0.3682, Accuracy: 0.8333


In [40]:
multiclass_model.eval()
with torch.no_grad():
    multiclass_predictions = multiclass_model(X_test)
    multiclass_predictions_labels = torch.argmax(multiclass_predictions, dim=1)

In [41]:
multiclass_accuracy = accuracy_score(y_test, multiclass_predictions_labels)
print(multiclass_accuracy)
multiclass_rep = classification_report(y_test, multiclass_predictions_labels)
print(multiclass_rep)

0.8235147279081378
              precision    recall  f1-score   support

           0       0.64      0.12      0.21        56
           1       0.67      0.73      0.70      2158
           2       0.89      0.87      0.88      5798

    accuracy                           0.82      8012
   macro avg       0.73      0.57      0.59      8012
weighted avg       0.83      0.82      0.82      8012



Our multiclass neural network performs similarly to the binary classification network at predicting cancellations.

It has an overall accuracy of 82%, meaning that 82% of all the predictions were correct.
The precision in row `1` tells us that when our model predicts a cancellation, it is correct 67% of the time. 
The recall score in row `1` tells us that our model captures 73% of the actual cancellations in our data.

Unfortunately, the model doesn't do the best job of predicting whether or not the customer will no-show. 

For no-shows (row class `0`), the precision score tells us that when our model predicts a no-show it is correct 64% of the time which is surprising well.
However, the low recall score tells us that our model only captures 12% of actual no-shows which is not very good. The lower recall score brings the F1 score down to 21% which indicates a not-so-great balance between precision and recall. This means that the model doesn't predict many no-shows and will most likely not be able to capture most customers who no-show in real-life. 

If our goal is to be able to reach out to potential no-shows, the low recall score is concerning. However, this all may be due to the low number of no-shows in the dataset: it is much harder for our model to find patterns predicting a no-show without more data. However, unlike the binary model, the multiclass does make an attempt to classify no-shows while still being able to predict cancellations ahead of time with similar performance.

In future research, we could improve the model by performing a more in-depth analysis of the features and doing a more robust feature selection process. Some examples might include collecting weather data at the time of each booking, reservations made on major holidays, economic conditions, or even global pandemics and health concerns.

Furthermore, we could also try to improve performance by modifying the neural network architecture like changing the number of nodes across the hidden layers, trying out different activation functions and optimizers, adding more hidden layers, or training on additional epochs, etc.